# Supervised CLS Token Classifier using Pretrained ViT(MAE) Encoder as Feature Extractor

## Data-Structure:

.\CLS_Dataset
├── test
│   ├── Fake
│   └── Real
├── train
│   ├── Fake
│   └── Real
└── val
    ├── Fake
    └── Real

### Imports

In [ ]:
import torch
import torch.nn as nn
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import os
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report


### Device Check

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple GPU
    print("Using MPS (Apple GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
    print("Using CUDA")
else:
    device = torch.device("cpu")   # fallback
    print("Using CPU")

### Encoder Loading and Head Swapping

In [ ]:
model_path = "/home/le.song/mae_fidd_model"
model = ViTForImageClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label={0: "Real", 1: "Fake"},
    label2id={"Real": 0, "Fake": 1},
    ignore_mismatched_sizes=True # Drops the MAE decoder, adds CLS head
).to(device)

### Freezing Encoder

In [ ]:
for name, param in model.vit.named_parameters():
    param.requires_grad = False

### Pre-processor

In [ ]:
processor = ViTImageProcessor.from_pretrained("facebook/vit-mae-base")

### Transforms

In [ ]:
def transform(image):
    return processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

### Data Loaders

In [ ]:
train_ds = ImageFolder(root="/home/le.song/CLS_Dataset/train", transform=transform) # Training now includes Fake images, splits are R:1500, GAN:500, IFv1:500, Diffusion:500, 2 subfolders/classes
val_ds = ImageFolder(root="/home/le.song/CLS_Dataset/val", transform=transform)# Val contains: R:750, GAN:250, IFv1:250, Diffusion:250, 2 subfolders/classes
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_ds = ImageFolder(root="/home/le.song/CLS_Dataset/test", transform=transform)# Test contains R:750, Gan:250, IFv1:250, Diffusion:250
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

### Optimizer

In [ ]:
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss() #Now using CEL because supervised

### Early Stopper Patience = 5

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

### Training Loop

In [ ]:
from sklearn.metrics import classification_report
import time

def train_classifier(model, train_loader, val_loader, optimizer, criterion, device, num_epochs=20):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)
    early_stopper = EarlyStopping(patience=5)
    # History for plotting
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    save_path = "/home/le.song/cls_classifier_model"
    os.makedirs(save_path, exist_ok=True)
    best_val_loss = float('inf')

    print(f"--- Starting Supervised Training (Linear Probing) ---")

    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training Phase
        model.train()
        train_loss, correct_train, total_train = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.logits, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100 * correct_train / total_train

        # Validation Phase
        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        y_true, y_pred = [], []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(pixel_values=images)
                loss = criterion(outputs.logits, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.logits, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predicted.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct_val / total_val
        
        # Logging
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{num_epochs}] ({epoch_time:.1f}s) | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # Save to history for loss curve
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(val_acc)

        scheduler.step(avg_val_loss)

        # Save Best Model I understand the encoder is frozen and I likely won't use this classifer again
        # However, in case something goes wrong and crashes I can load from checkpoint.
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            model.save_pretrained(save_path)
            print(f"  --> Saved new best model (Val Loss: {best_val_loss:.4f})")

        # Check Early Stopping
        early_stopper(avg_val_loss)
        if early_stopper.early_stop:
            print("Early stopping triggered. Training halted.")
            break

    # Final Classification Report
    print("\n" + "="*30)
    print("FINAL CLASSIFICATION REPORT")
    print("="*30)
    print(classification_report(y_true, y_pred, target_names=["Real", "Fake"]))
    
    return model, history

### Loss Curve

In [ ]:
def plot_training_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    
    plt.figure(figsize=(12, 5))

    # Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], 'b-o', label='Training Loss')
    plt.plot(epochs, history["val_loss"], 'r-o', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Accuracy Curve
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["val_acc"], 'g-s', label='Validation Accuracy')
    plt.title('Validation Accuracy (%)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("supervised_training_curves.png")
    print("Training curves saved as 'supervised_training_curves.png'")
    plt.show()

### Testing Logic

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_classifier(model_path, test_loader, device):
    print(f"--- Loading Best Model from {model_path} ---")
    # Load the specific fine-tuned weights
    model = ViTForImageClassification.from_pretrained(model_path).to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    print(f"--- Evaluating on {len(test_loader.dataset)} Test Images ---")
    
    with torch.no_grad():
        for batch in test_loader:
            images, labels = batch[0].to(device), batch[1].to(device) # Ignores paths
            outputs = model(pixel_values=images)
            
            # Get the binary prediction (0: Real, 1: Fake)
            _, preds = torch.max(outputs.logits, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    
    print("\n" + "="*45)
    print("      SUPERVISED TEST PERFORMANCE")
    print("="*45)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("\nDetailed Report:")
    print(classification_report(all_labels, all_preds, target_names=["Real", "Fake"]))
    
    return all_labels, all_preds

### Logging Function

In [ ]:
def save_summary(history, report, save_path="supervised_summary.txt"):
    with open(save_path, "w") as f:
        f.write("MAE SUPERVISED CLASSIFICATION SUMMARY\n")
        f.write("=====================================\n")
        f.write(f"Best Val Accuracy: {max(history['val_acc']):.2f}%\n")
        f.write(f"Final Val Loss:    {history['val_loss'][-1]:.4f}\n")
        f.write("\nDetailed Classification Report:\n")
        f.write(report)
    print(f"Summary saved to {save_path}")

### Probability Distribution Plot

In [ ]:
def plot_probability_distribution(model, dataloader, device):
    model.eval()
    real_probs = []
    fake_probs = []

    with torch.no_grad():
        for batch in dataloader:
            images, labels = batch[0].to(device), batch[1].to(device) # Ignores paths
            outputs = model(pixel_values=images)
            # Apply Softmax to get 0.0-1.0 probabilities
            probs = torch.softmax(outputs.logits, dim=1)[:, 1] 
            
            for p, l in zip(probs, labels):
                if l == 0: real_probs.append(p.item())
                else: fake_probs.append(p.item())

    plt.figure(figsize=(10, 6))
    plt.hist(real_probs, bins=50, alpha=0.5, label='Real', color='blue')
    plt.hist(fake_probs, bins=50, alpha=0.5, label='Fake', color='red')
    plt.title("Prediction Confidence Distribution (Supervised)")
    plt.xlabel("Probability of being 'Fake'")
    plt.ylabel("Frequency")
    plt.legend()
    plt.savefig("probability_distribution.png")

### Heatmap Generator

In [ ]:
def generate_supervised_heatmaps(model, dataloader, device, num_real=5, num_fake=5, save_prefix="Stage2"):
    model.eval()
    os.makedirs("supervised_heatmaps", exist_ok=True)

    print(f"--- Generating {num_real} Real and {num_fake} Fake Supervised Heatmaps ---")

    count_real = 0
    count_fake = 0
    idx_to_class = {v: k for k, v in dataloader.dataset.class_to_idx.items()}
    with torch.no_grad():
        for images, labels, paths in dataloader:
            label_idx = labels[0].item()
            label_text = idx_to_class[label_idx] # "Fake" or "Real"
            is_fake = (label_text.lower() == 'fake')
            
            # Skip logic to get a balanced set of images
            if is_fake and count_fake >= num_fake: continue
            if not is_fake and count_real >= num_real: continue

            images = images.to(device)
            # We need output_attentions=True to see what the CLS token is looking at
            outputs = model(pixel_values=images, output_attentions=True)

            # Get Prediction and Confidence
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            conf, pred_idx = torch.max(probs, dim=1)
            pred_text = idx_to_class[pred_idx[0].item()]
            confidence = conf[0].item() * 100
            
            # 1. Extract Attention Rollout (Last Layer)
            # Shape: [Batch, Heads, Seq, Seq]. Seq = 1 (CLS) + 196 (Patches)
            attentions = outputs.attentions[-1] 
            # Average across all heads, take CLS token's attention to all other patches
            cls_attn = attentions[0].mean(dim=0)[0, 1:].view(14, 14).cpu().numpy()

            # 2. Normalize and Resize Attention for Overlay
            # Attention values are very small, so we normalize 0-1 for visualization
            cls_attn_norm = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
            heatmap_resized = np.array(Image.fromarray(cls_attn_norm.astype(np.float32)).resize((224, 224), resample=Image.BICUBIC))

            # --- IMAGE PREP ---
            # Process the first image in the batch for visualization
            orig = images[0].cpu().permute(1, 2, 0).numpy()
            orig = (orig - orig.min()) / (orig.max() - orig.min() + 1e-8)

            # 3. plotting
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            
            # Subplot 0: Original + Verdict
            # Green if correct, Red if wrong
            title_color = "green" if pred_text == label_text else "red"
            axes[0].imshow(orig)
            axes[0].set_title(f"GT: {label_text}\nPRED: {pred_text} ({confidence:.1f}%)", color=title_color)
            axes[0].axis('off')

            # Subplot 1: Raw Attention Map
            axes[1].imshow(cls_attn, cmap='viridis')
            axes[1].set_title("CLS Attention (14x14)")
            axes[1].axis('off')

            # Subplot 2: Heatmap Overlay
            axes[2].imshow(orig) 
            im = axes[2].imshow(heatmap_resized, cmap='jet', alpha=0.5) 
            axes[2].set_title(f"Decision Hotspots")
            axes[2].axis('off')

            plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

            # 4. Save the result
            clean_name = os.path.basename(paths[0]).split('.')[0]
            save_path = f"supervised_heatmaps/{save_prefix}_{label_text}_{clean_name}.png"
            plt.savefig(save_path, bbox_inches='tight')
            plt.close()

            if is_fake: count_fake += 1
            else: count_real += 1
            
            if count_real >= num_real and count_fake >= num_fake: break

    print(f"Supervised heatmaps saved in 'supervised_heatmaps/'")

### Results Getter for Heatmaps

In [ ]:
# Helper to get paths for the Heatmap Generator
class ImageFolderWithPaths(ImageFolder):
    def __getitem__(self, index):
        original_tuple = super(ImageFolderWithPaths, self).__getitem__(index)
        path = self.imgs[index][0]
        tuple_with_path = (original_tuple + (path,))
        return tuple_with_path

# Execution Block

In [ ]:
if __name__ == "__main__":
    # 1. Setup Data Loaders (Using the Path-aware version for the test set)
    print("--- Preparing Data ---")
    train_ds = ImageFolder(root="/home/le.song/CLS_Dataset/train", transform=transform)
    val_ds = ImageFolder(root="/home/le.song/CLS_Dataset/val", transform=transform)
    test_ds = ImageFolderWithPaths(root="/home/le.song/CLS_Dataset/test", transform=transform)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False) # BS=1 is better for path-tracking in heatmaps

    # 2. Run Training
    trained_model, history = train_classifier(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        optimizer=optimizer, 
        criterion=criterion, 
        device=device,
        num_epochs=20
    )
    
    # Loss & Accuracy Curves
    plot_training_curves(history)

    # 3. Final Evaluation on Test Set
    best_model_path = "/home/le.song/cls_classifier_model"
    y_true, y_pred = evaluate_classifier(best_model_path, test_loader, device)

    # 4. Save Final Summary Report
    final_report = classification_report(y_true, y_pred, target_names=["Real", "Fake"])
    save_summary(history, final_report, save_path="supervised_test_summary.txt")

    # 5. Visualizations

    # Probability Distribution (Confidence Histogram)
    plot_probability_distribution(trained_model, test_loader, device)
    
    # Decision Hotspots (Attention Heatmaps)
    generate_supervised_heatmaps(
        model=trained_model, 
        dataloader=test_loader, 
        device=device, 
        num_real=5, 
        num_fake=5
    )

    print("\n[STAGES 2 COMPLETE] All results, models, and heatmaps have been saved.")